# 08. Deployment & Monitoring

## 📚 Learning Objectives

By completing this notebook, you will:
- Wrap a model in a simple API (e.g. Flask/FastAPI)
- Validate inputs and return predictions
- Run and test the API locally

## 🔗 Where this fits

**Builds on:** Course 04 (AIAT 114) — Unit 5 (a tuned, selected model) and Course 05 — Unit 4 — a model nobody can call is not a product yet.

**Used later in:** Course 11 (AIAT 125) — Units 1-2, which take this Flask/FastAPI sketch to production serving, versioning and containers.

---


## 🎯 The case: $440 million in 45 minutes, from a deployment

**1 August 2012.** Knight Capital Group deployed new code to its automated trading system,
SMARS, ahead of the NYSE's Retail Liquidity Program going live. The new code reused a flag
that an old, dormant function called **"Power Peg"** had used years earlier. A technician
copied the release to **seven of the firm's eight servers**. The eighth kept the old code.

At 09:30 the market opened. Orders routed to the eighth server triggered the dead Power Peg
logic, which sent child orders without ever registering that the parent order was filled —
so it kept sending them. In about **45 minutes** Knight executed 4 million trades across 397
million shares, moved prices in 148 NYSE-listed stocks, and lost roughly **$440 million
pre-tax**. The firm was acquired within the year.

No model was wrong. No data was dirty. A deployment put one version of the code somewhere
it should not have been, and nothing in the system noticed.

**What goes wrong without this lesson.** Everything you are about to build exists because of
that gap: **versioning** (which artifact is running where), **metadata** (what was it trained
on, and how good was it), **input validation** (reject what you cannot handle), **logging**
(what did it actually do), and **monitoring** (is it still behaving). You will pickle a model
that scored **R² 0.5404, RMSE $78,785** and write that number into
`model_metadata.json` — because, as the notebook says, a deployed model whose accuracy
nobody wrote down cannot be monitored.


## The Story

**BEFORE**: You can build ML models but don't know how to deploy them for real-world use.

**AFTER**: You'll learn model deployment: APIs, containers, cloud deployment, and making models accessible to users!

**Why this matters**: Deployment & Monitoring is essential for building complete, professional data science solutions!

---

# Unit 5 - Example 08: Deployment & Monitoring

## 🔗 Solving the Problem from Example 07

**Remember the dead end from Example 07?**
- We learned large dataset handling strategies
- But the model is ready - how do we deploy it for users?
- We needed deployment and monitoring strategies

**This notebook solves that problem!**
- We'll learn **model deployment strategies**
- We'll learn **monitoring and maintenance**
- We'll learn **production deployment best practices**

**This solves the deployment problem from Example 07!**

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- **Real data:** the California Housing dataset — 20,640 census block groups from
  the 1990 U.S. census (5,000 sampled). We train the model that then gets pickled,
  wrapped in a function, served over Flask and FastAPI, and monitored.
- sklearn, pickle, json, Flask / FastAPI (optional)

**Outputs:** What you'll see when you run the cells

- `deployed_model.pkl` and `model_metadata.json` in this folder
- Live HTTP responses from the two toy APIs, predicting real house values

---


In [1]:
# Step 1: Import necessary libraries
import numpy as np
import pandas as pd
import json
import pickle
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("=" * 70)
print("Example 08: Deployment & Monitoring")
print("=" * 70)
print("\n📚 Prerequisites: Examples 02-07 completed, deployment knowledge")
print("🔗 This is the EIGHTH example in Unit 5 - deployment and monitoring")
print("🎯 Goal: Master deploying and monitoring ML models")
print("Reference: Study 19.pdf before running this code example.\n")

Example 08: Deployment & Monitoring

📚 Prerequisites: Examples 02-07 completed, deployment knowledge
🔗 This is the EIGHTH example in Unit 5 - deployment and monitoring
🎯 Goal: Master deploying and monitoring ML models
Reference: Study 19.pdf before running this code example.



# 08. TRAIN MODEL FOR DEPLOYMENT

In [2]:
# WHAT: Train and evaluate the regression model we will deploy, on real census data.
# WHY: Deployment starts from a validated artifact - never ship a model whose metrics you have not checked.

print("\n1. Training Model for Deployment")
print("-" * 70)

from sklearn.datasets import fetch_california_housing

# Real data: 1990 U.S. census block groups. Target = median house value ($100,000s).
housing = fetch_california_housing(as_frame=True)
house = housing.frame.sample(n=5000, random_state=42)
FEATURE_NAMES = ['MedInc', 'HouseAge', 'AveRooms']
X = house[FEATURE_NAMES].to_numpy()
y = house['MedHouseVal'].to_numpy()
print(f"Real dataset: {len(X):,} California census block groups")
print(f"Features deployed: {FEATURE_NAMES}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\n✓ Model trained successfully")
print(f"MSE: {mse:.4f}, R²: {r2:.4f}")
print(f"RMSE in dollars: ${mse ** 0.5 * 100000:,.0f}")
print(f"\n💡 R² = {r2:.2f} on three real features. Ship that number in the metadata:")
print("   a deployed model whose accuracy nobody wrote down cannot be monitored.")


1. Training Model for Deployment
----------------------------------------------------------------------
Real dataset: 5,000 California census block groups
Features deployed: ['MedInc', 'HouseAge', 'AveRooms']

✓ Model trained successfully
MSE: 0.6207, R²: 0.5404
RMSE in dollars: $78,785

💡 R² = 0.54 on three real features. Ship that number in the metadata:
   a deployed model whose accuracy nobody wrote down cannot be monitored.


# 2. SAVE MODEL FOR DEPLOYMENT


In [3]:
# WHAT: Serialize the model with pickle and write version/metrics metadata to JSON.
# WHY: The saved file is what production loads; the metadata says which version it is and how good it was at training time.

print("\n\n2. Saving Model")
print("-" * 70)
# Save model using pickle
model_path = 'deployed_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model, f)
print(f"✓ Model saved to {model_path}")
# Save model metadata
metadata = {
    'model_version': '1.0.0',
    'deployed_at': datetime.now().isoformat(),
    'training_date': datetime.now().isoformat(),
    'training_data': 'California Housing (1990 U.S. census), 5000-block sample',
    'target': 'median house value in $100,000s',
    'metrics': {
        'mse': float(mse),
        'r2': float(r2)
    },
    'features': FEATURE_NAMES,
    'model_type': 'LinearRegression'
}
metadata_path = 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Model metadata saved to {metadata_path}")
print(json.dumps(metadata, indent=2))



2. Saving Model
----------------------------------------------------------------------
✓ Model saved to deployed_model.pkl
✓ Model metadata saved to model_metadata.json
{
  "model_version": "1.0.0",
  "deployed_at": "2026-08-25T22:20:44.137375",
  "training_date": "2026-08-25T22:20:44.137379",
  "training_data": "California Housing (1990 U.S. census), 5000-block sample",
  "target": "median house value in $100,000s",
  "metrics": {
    "mse": 0.6207108731580728,
    "r2": 0.5404326975955127
  },
  "features": [
    "MedInc",
    "HouseAge",
    "AveRooms"
  ],
  "model_type": "LinearRegression"
}


# 3. DEPLOYMENT FUNCTION


In [4]:
# WHAT: Wrap prediction in a function with input validation, logging, and error handling.
# WHY: Production predictions fail on bad input - validate first, log always, and raise informative errors.

print("\n\n3. Deployment Function")
print("-" * 70)
def predict(model, features):
    """
    Make predictions using deployed model
    """
    try:
        logger.info(f"Making prediction for {len(features)} samples")
        # Validate input
        if len(features.shape) != 2:
            raise ValueError("Features must be 2D array")
        # Make prediction
        prediction = model.predict(features)
        logger.info(f"Prediction successful: {len(prediction)} results")
        return prediction
    except Exception as e:
        logger.error(f"Prediction failed: {str(e)}", exc_info=True)
        raise e

# Test deployment function
test_features = X_test[:5]
predictions = predict(model, test_features)
print(f"\n✓ Deployment function tested successfully")
print(f"Sample predictions: {predictions[:3]}")

2026-08-25 22:20:44,141 - INFO - Making prediction for 5 samples


2026-08-25 22:20:44,141 - INFO - Prediction successful: 5 results




3. Deployment Function
----------------------------------------------------------------------

✓ Deployment function tested successfully
Sample predictions: [1.14384403 1.52189487 2.33340918]


### 3b. API Deployment with Flask or FastAPI

Below we define minimal REST APIs. Install if needed: `pip install flask fastapi uvicorn`.

In [5]:
# WHAT: Serve the model over HTTP with a Flask /predict endpoint and test it live.
# WHY: A REST API decouples the model from its callers - any language can POST JSON and get predictions.

# Flask API example: serve predictions via REST
print("\n3b. API Deployment (Flask / FastAPI)")
print("-" * 70)
try:
    from flask import Flask, request, jsonify
    import threading
    import time
    
    app = Flask(__name__)
    
    @app.route("/predict", methods=["POST"])
    def predict_api():
        """Serve model predictions. POST JSON: {\"features\": [[f1,f2,f3], ...]}"""
        data = request.get_json()
        if not data or "features" not in data:
            return jsonify({"error": "Expected JSON with 'features' key"}), 400
        X = np.array(data["features"])
        preds = model.predict(X)
        return jsonify({"predictions": preds.tolist()})
    
    def run_flask():
        app.run(host="127.0.0.1", port=5000, use_reloader=False, threaded=True)
    
    thread = threading.Thread(target=run_flask, daemon=True)
    thread.start()
    time.sleep(1.5)
    
    import urllib.request
    body = json.dumps({"features": X_test[:3].tolist()}).encode()
    req = urllib.request.Request("http://127.0.0.1:5000/predict", data=body, method="POST", headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as r:
        out = json.loads(r.read().decode())
    print("✓ Flask /predict response:", out)
except ImportError:
    print("⚠ Install Flask to run API example: pip install flask")
except Exception as e:
    print(f"⚠ Flask demo skipped: {e}")


3b. API Deployment (Flask / FastAPI)
----------------------------------------------------------------------
 * Serving Flask app '__main__'


 * Debug mode: off


2026-08-25 22:20:44,199 - INFO - WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000


2026-08-25 22:20:44,199 - INFO - Press CTRL+C to quit


2026-08-25 22:20:45,712 - INFO - 127.0.0.1 - - [25/Aug/2026 22:20:45] "POST /predict HTTP/1.1" 200 -


✓ Flask /predict response: {'predictions': [1.1438440306405422, 1.5218948699718378, 2.3334091783170794]}


## 💬 Discuss

The deployed model predicts California house values from three features and scores **R²
0.5404, MSE 0.6207, RMSE $78,785**. It is served over Flask and FastAPI, and every
prediction is logged.

1. **RMSE $78,785** on a target whose median is about $180,000. Write the sentence that goes
   in the API documentation so a downstream engineer cannot mistake a prediction for a
   valuation. What HTTP response would you add to make that limit impossible to ignore?
2. Flask accepted whatever JSON it was given; FastAPI validates against a pydantic type.
   Take one concrete bad input — a string where a float belongs, a missing field, a
   negative number of rooms — and trace what each version does with it. Which failure is
   worse: a 422 error, or a confident prediction from garbage?
3. Knight Capital's fault was that one of eight servers ran old code. Your model artifact is
   `deployed_model.pkl` with a version in a JSON file beside it. Describe how you would find
   out, in production, that one instance is serving version 1.0.0 while the rest serve
   1.1.0 — and how long it would take you today.


In [6]:
# WHAT: Repeat the API with FastAPI + pydantic validation.
# WHY: FastAPI validates request shapes from type hints - malformed input is rejected before reaching the model.

# FastAPI example: same idea, async-friendly API
try:
    import threading
    import time
    import urllib.request
    from typing import List
    from fastapi import FastAPI
    from pydantic import BaseModel
    import uvicorn

    api = FastAPI(title="ML Predictions")

    class FeaturesRequest(BaseModel):
        features: List[List[float]]

    @api.post("/predict")
    def predict_endpoint(req: FeaturesRequest):
        X = np.array(req.features)
        preds = model.predict(X)
        return {"predictions": preds.tolist()}

    def run_fastapi():
        uvicorn.run(api, host="127.0.0.1", port=8000, log_level="warning")

    thread2 = threading.Thread(target=run_fastapi, daemon=True)
    thread2.start()
    time.sleep(1.5)

    body2 = json.dumps({"features": X_test[:2].tolist()}).encode()
    req2 = urllib.request.Request("http://127.0.0.1:8000/predict", data=body2, method="POST", headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req2) as r2:
        out2 = json.loads(r2.read().decode())
    print("✓ FastAPI /predict response:", out2)
except NameError:
    pass  # threading, urllib, etc. from Flask cell
except ImportError:
    print("⚠ Install FastAPI + uvicorn to run: pip install fastapi uvicorn")
except Exception as e:
    print(f"⚠ FastAPI demo skipped: {e}")

✓ FastAPI /predict response: {'predictions': [1.1438440306405422, 1.5218948699718378]}


4. MONITORING SETUP


In [7]:
# WHAT: Build a ModelMonitor class that logs predictions and errors.
# WHY: Deployment is not the end - logged predictions are the raw material for drift detection (next example).

print("\n\n4. Monitoring Setup")
print("-" * 70)
class ModelMonitor:
    """Simple model monitoring class"""
    def __init__(self):
        self.predictions_log = []
        self.errors_log = []
    
    def log_prediction(self, features, prediction, actual=None):
        """Log prediction for monitoring"""
        log_entry = {
            'timestamp': datetime.now().isoformat(), 'features': features.tolist() if isinstance(features, np.ndarray) else features,
            'prediction': float(prediction) if np.isscalar(prediction) else prediction.tolist(), 'actual': float(actual) if actual is not None and np.isscalar(actual) else None
        }
        self.predictions_log.append(log_entry)
        logger.info(f"Logged prediction: {log_entry['prediction']}")
    
    def log_error(self, error_message):
        """Log error for monitoring"""
        error_entry = {
            'timestamp': datetime.now().isoformat(), 'error': error_message
        }
        self.errors_log.append(error_entry)
        logger.error(f"Logged error: {error_message}")
    
    def get_stats(self):
        """Get monitoring statistics"""
        return {
            'total_predictions': len(self.predictions_log), 'total_errors': len(self.errors_log),
            'latest_prediction': self.predictions_log[-1] if self.predictions_log else None
        }

monitor = ModelMonitor()
# Log five real predictions through the monitor
for i in range(5):
    features = X_test[i:i+1]
    pred = model.predict(features)[0]
    actual = y_test.iloc[i] if hasattr(y_test, "iloc") else y_test[i]
    monitor.log_prediction(features[0], pred, actual)

stats = monitor.get_stats()
print(f"\nMonitoring Statistics:")
print(f"  Total predictions: {stats['total_predictions']}")
print(f"  Total errors: {stats['total_errors']}")

2026-08-25 22:20:47,363 - INFO - Logged prediction: 1.1438440306405422


2026-08-25 22:20:47,363 - INFO - Logged prediction: 1.5218948699718378


2026-08-25 22:20:47,364 - INFO - Logged prediction: 2.3334091783170794


2026-08-25 22:20:47,364 - INFO - Logged prediction: 1.3223071459254048


2026-08-25 22:20:47,364 - INFO - Logged prediction: 3.3218679048601105




4. Monitoring Setup
----------------------------------------------------------------------

Monitoring Statistics:
  Total predictions: 5
  Total errors: 0


# 5. DEPLOYMENT CHECKLIST


In [8]:
# WHAT: Print the deployment checklist.
# WHY: Each unchecked item is a production incident waiting to happen.

print("\n\n5. Deployment Checklist")
print("-" * 70)
checklist = {
'Model trained and validated': True, 'Model saved and versioned': True,
'Metadata documented': True,
'Error handling implemented': True,
'Logging configured': True,
'Monitoring set up': True,
'Documentation created': True
}
print("\nDeployment Checklist:")
for item, status in checklist.items():
    status_symbol = "✓" if status else "✗"
    print(f"  {status_symbol} {item}")



5. Deployment Checklist
----------------------------------------------------------------------

Deployment Checklist:
  ✓ Model trained and validated
  ✓ Model saved and versioned
  ✓ Metadata documented
  ✓ Error handling implemented
  ✓ Logging configured
  ✓ Monitoring set up
  ✓ Documentation created


# 6. SUMMARY


In [9]:
# WHAT: Print the summary.
# WHY: Serialize, serve, validate, log, monitor - the five verbs of model deployment.

print("\n\n" + "=" * 70)
print("Summary")
print("=" * 70)
print("\nKey Concepts Covered:")
print("1. Model serialization and saving")
print("2. Deployment functions")
print("3. Monitoring and logging")
print("4. Version control")
print("5. Deployment best practices")
print("\n" + "=" * 70)
print("Example 08 complete - Unit 5 continues!")
print("Next Steps: Example 09 (Model Monitoring), then Example 10 closes the course")
print("=" * 70)



Summary

Key Concepts Covered:
1. Model serialization and saving
2. Deployment functions
3. Monitoring and logging
4. Version control
5. Deployment best practices

Example 08 complete - Unit 5 continues!
Next Steps: Example 09 (Model Monitoring), then Example 10 closes the course


## ⚠️ Where this breaks

- **`app.run()` is a development server and Flask says so in the output above.** It is
  single-threaded, has no request limits, no TLS and no process supervision. Production
  needs a WSGI/ASGI server (gunicorn, uvicorn) behind a reverse proxy — and that is before
  containers, health checks and autoscaling.
- **Pickle is version-fragile and unsafe.** A model pickled with one scikit-learn version
  may not load in the next, and unpickling untrusted bytes executes arbitrary code. Pin the
  library versions in the metadata; for anything long-lived, prefer ONNX or an explicit
  serialisation format.
- **Input validation checks types, not meaning.** FastAPI will happily accept `MedInc =
  900` or `AveRooms = 0.1`. Both are valid floats and neither resembles a California block
  group. Range checks and distribution checks against the training data are a separate,
  necessary layer.
- **Logging predictions is not monitoring.** The `ModelMonitor` above counts predictions and
  errors. It cannot see accuracy, because in production the true label arrives later or
  never. Lesson 09 is where that gap gets addressed, and it is the harder problem.
- **The assumption that must hold: the request distribution resembles the training data.**
  This model was trained on the **1990 census** with the target censored at $500,001 (Unit
  2 lesson 06). It cannot predict above the cap, it knows nothing about 2026, and the API
  gives no hint of either.
- **Deployment is a systems problem, not an ML problem.** Rollback, blue/green or canary
  release, artifact provenance, config management and secrets are where deployments actually
  fail — Knight Capital lost $440 million to a file-copy step. Sculley et al. (2015) call
  this the hidden technical debt; Course 11 is where the diploma treats it properly.


## 📚 References

1. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
2. Paleyes, A., Urma, R.-G., & Lawrence, N. D. (2022). *Challenges in Deploying Machine Learning: A Survey of Case Studies*. ACM Computing Surveys, 55(6), 1-29. <https://arxiv.org/abs/2011.09926>
3. Kreuzberger, D., Kühl, N., & Hirschl, S. (2022). *Machine Learning Operations (MLOps): Overview, Definition, and Architecture*. IEEE Access, 11, 31866-31879. <https://arxiv.org/abs/2205.02302>